# Máy AI cá nhân StoryForge · Viết truyện

1. Chọn runtime **GPU** hoặc **CPU**.
2. Bấm **▶** ở ô **Máy AI cá nhân StoryForge · kết nối bằng mã xác nhận** ngay bên dưới. Bấm **Sao chép mã**, dán vào **Cài đặt → Máy AI cá nhân** của StoryForge và xác nhận.
3. Chờ thông báo **sẵn sàng**, rồi quay lại StoryForge để chat/viết truyện (ô tự dừng vòng xoay để không khóa thao tác Colab nhưng máy AI vẫn chạy nền).

<details>

<summary><strong>Hướng dẫn chi tiết và xử lý sự cố · bấm để xem</strong></summary>

Đây là mẫu công khai từ GitHub, có sẵn mã và biểu mẫu để chạy. Mở mẫu không tự tạo file trong Drive.  
Bạn có thể chọn **Tệp → Lưu bản sao vào Drive** để giữ chỉnh sửa; bước này **không bắt buộc** mỗi ngày.  
Chỉnh sửa trong phiên hoặc bản sao của bạn không sửa nguồn GitHub nếu bạn không có quyền ghi vào kho.  
Mỗi người chạy runtime và liên kết riêng, không dùng chung GPU, key hoặc tunnel của người khác.  
Phiên runtime mới không mang theo model đã tải, RAM, tiến trình hay credential của phiên cũ.

### Bắt đầu

1. Chọn runtime GPU hoặc CPU, mở notebook rồi chạy ô điều khiển.  
2. Chọn **Ghép nối / khởi động**. Notebook tạo mã dùng một lần và chờ tối đa 10 phút.  
3. Trong StoryForge, mở **Cài đặt → Máy AI cá nhân**, nhập mã và xác nhận kết nối.  
4. Khi trạng thái web và notebook đều báo sẵn sàng, StoryForge tự chọn đúng máy vừa liên kết khi không có lượt sinh/Auto Write. Bấm **Thử kết nối** để kiểm tra Direct; **Dùng máy này** cho phép thử chọn lại khi cần.  
5. Chạy lại cùng ô sẽ tiếp tục phiên đang chờ hoặc đang hoạt động; không tạo pairing khác.

### Khi tải GGUF bị kẹt hoặc đã bấm Dừng

Sau bước cài thư viện, notebook hiển thị riêng bước tải công cụ GPU/CPU từ GitHub.
Nếu nguồn tải công cụ trả lỗi, thông báo sẽ nêu tên file và mã HTTP; lỗi mạng tạm thời được thử tối đa 3 lần.
Lượt tải GGUF chạy trong tiến trình riêng; bấm Dừng sẽ dừng cả tiến trình tải GGUF do notebook tạo.
Nếu không lấy được khóa file sau 60 giây, notebook báo rõ thay vì chờ vô hạn.
Tải quá 30 phút sẽ dừng để kiểm tra lại mạng; trong lúc chờ có thông báo tiến độ tải và tốc độ định kỳ.
Không tự xóa khóa hoặc file model đã tải. Khi chạy lại, Hugging Face dùng cache hoặc tiếp tục phần tải còn dùng được.
Nếu bản notebook cũ đã kẹt và tải lại trang vẫn không hết, khởi động lại phiên chạy Colab
(Runtime → Restart session), rồi ghép nối lại. Thao tác này dừng máy AI hiện tại;
tải lại trang trình duyệt hoặc kết nối lại đơn thuần không khởi động lại máy Colab.

Notebook không yêu cầu Colab Secrets và không nhúng StoryForge session token, Cloudflare API key hoặc credential mẫu.
Sau khi xác nhận, StoryForge cấp API key AI, control secret và connector token một lần. Chúng chỉ ở bộ nhớ runtime;
gateway chỉ nhận API key và đường dẫn tới file lease không chứa secret. Không chia sẻ runtime hoặc output của notebook.

StoryForge cấp hostname riêng và remote tunnel có ingress loopback tới gateway `127.0.0.1:18081`.
Notebook không dùng hostname cá nhân cố định và không mở cổng model `18080` ra Internet. Thu hồi hoặc hết lease sẽ đóng gateway/tunnel;
stream đang chạy bị dừng fail-closed. Nếu runtime mất mạng, lease cũ tối đa 120 giây; heartbeat 60 giây; phiên tối đa 8 giờ.

Mặc định model là `HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive` / `Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf`,
Các lựa chọn Q/IQ ghim revision `45b6a334b4bcd1d7f37179df58b3b1d66a184e5d`, thiết bị **Tự động**, context **128000 mỗi lượt**, max output **64000**, **3 slot**. GGUF khác vẫn dùng ba trường nhập riêng. Chọn trong form chưa tải model; chỉ tải/nạp khi chủ động chạy. Đây là giá trị cấu hình, không phải benchmark.
Chưa có bằng chứng inference thực tế đạt đủ 128000/64000 trên Colab; khả năng chạy tùy RAM/VRAM và thời lượng phiên.

Để tạo lại mã sau khi thu hồi: thu hồi máy cũ trong StoryForge, đợi lease hết hạn (tối đa 120 giây), rồi chọn **Tạo mã liên kết mới**.
Notebook không tự nhận lại credential nếu claim response bị mất; khi đó phải tạo pairing mới. Không tự retry request inference.

</details>


In [ ]:
#@title Máy AI cá nhân StoryForge · kết nối bằng mã xác nhận
#@markdown Chọn **Ghép nối / khởi động**, bấm **▶ Chạy**, rồi nhập mã hiện ra vào StoryForge và xác nhận.
thao_tac = "Ghép nối / khởi động" #@param ["Ghép nối / khởi động", "Khôi phục cấu hình trước", "Tạo mã liên kết mới", "Xem trạng thái", "Chẩn đoán", "Dừng máy chủ"]
#@markdown ### Model và thiết bị
model_AI = "Q8_K_P" #@param ["IQ3_M", "IQ4_XS", "Q2_K_P", "Q3_K_M", "Q3_K_P", "Q4_K_M", "Q4_K_P", "Q5_K_M", "Q5_K_P", "Q6_K_P", "Q8_K_P", "GGUF khác"]
thiet_bi = "Tự động" #@param ["Tự động", "GPU", "CPU"]
so_yeu_cau_dong_thoi = 3 #@param [1, 3, 5] {type:"raw"}
#@markdown **3 lượt đồng thời** mặc định; **5 là thử nghiệm**, **1** để quay về tải thấp. Hàng đợi tối đa 5, chờ tối đa 60 giây rồi từ chối. Tăng slot sẽ nạp lại model và tăng RAM/VRAM; không tự giảm cấu hình khi thiếu bộ nhớ.
#@markdown ---
#@markdown ### 🧠 CHẾ ĐỘ SUY LUẬN SÂU (THINKING) — BẬT ĐỂ AI NGHĨ KỸ & THÔNG MINH HƠN
che_do_thinking = "Tắt thinking (Nhanh · Mặc định)" #@param ["Tắt thinking (Nhanh · Mặc định)", "Bật thinking (Suy luận sâu · Thông minh hơn)"]
#@markdown > **Gợi ý:** **Mặc định Tắt (No Thinking)** để viết truyện mượt và phản hồi nhanh. Chuyển sang **Bật thinking** khi cần lập dàn ý, kiểm tra mâu thuẫn logic hoặc phân tích nhiều ràng buộc khó; sau khi đổi lựa chọn bấm **▶ Chạy lại ô** (sẽ nạp lại model khoảng 30–60 giây).
#@markdown ---
#@markdown ### Ngữ cảnh và độ dài trả lời
#@markdown **128000 context mỗi lượt / 64000 đầu ra tối đa mỗi lượt**. Pool context = context × số slot (3: 384000; 5: 640000). Context gồm đầu vào, lịch sử, hướng dẫn và đầu ra; không bắt mỗi yêu cầu sinh đủ giới hạn. Chưa benchmark đủ độ dài hoặc bộ nhớ trên GPU của bạn.
context_tokens = 128000 #@param {type:"integer"}
max_output_tokens = 64000 #@param {type:"integer"}
#@markdown ### GGUF
hf_repo = "HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive" #@param {type:"string"}
gguf_file = "Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf" #@param {type:"string"}
hf_revision = "main" #@param {type:"string"}
#@markdown Không cần Colab Secrets. Notebook chỉ nhận credential sau khi người dùng xác nhận mã trong StoryForge.
# File này do build_multiuser_notebook.py sinh; origin không thể đổi trong biểu mẫu.
import base64, io, zipfile, sys, importlib, importlib.metadata, subprocess
SF_ORIGIN = 'https://storyforge-ai-pilot.canhettg113.workers.dev'
if SF_ORIGIN.endswith('.invalid'):
    raise RuntimeError('REVIEW ONLY: origin mẫu .invalid chưa kết nối được tới StoryForge; không gọi mạng.')
try:
    _sf_httpx_version = importlib.metadata.version('httpx')
except importlib.metadata.PackageNotFoundError:
    _sf_httpx_version = None
if _sf_httpx_version != '0.28.1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'httpx==0.28.1'],
                   check=True, timeout=180)
from pathlib import Path
_sf_folder = Path('/content/storyforge-personal-ai/source')
_sf_folder.mkdir(parents=True, exist_ok=True)
_sf_payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_sf_payload))) as _sf_zip:
    _sf_zip.extractall(_sf_folder)
if str(_sf_folder) not in sys.path:
    sys.path.insert(0, str(_sf_folder))
import runtime, connection, settings, linked_connection, start_colab
for _sf_module in (runtime, connection, settings, linked_connection, start_colab):
    importlib.reload(_sf_module)
_sf_personal_state = globals().get('SF_STATE', {})
if isinstance(_sf_personal_state, dict) and any(
        _sf_personal_state.get(_sf_name) is not None
        and _sf_personal_state[_sf_name].poll() is None
        for _sf_name in ('model', 'gateway', 'tunnel')):
    raise RuntimeError('Dừng máy chủ notebook cá nhân trong runtime này trước khi ghép nối nhiều người dùng.')
start_colab.ROOT = Path('/content/storyforge-personal-ai')
SF_LINKED_STATE = globals().get('SF_LINKED_STATE', {})
SF_LINKED_STATE['multiuser'] = True
_sf_settings = {'model_choice': model_AI, 'device': thiet_bi,
    'parallel': so_yeu_cau_dong_thoi, 'thinking_mode': che_do_thinking,
    'context': context_tokens, 'max_tokens': max_output_tokens,
    'repo': hf_repo, 'filename': gguf_file, 'revision': hf_revision}

def _sf_pairing_card(code):
    from html import escape
    return r"""
<style>
.sf-pair {box-sizing:border-box; max-width:720px; padding:24px; margin:12px 0 20px;
  border-radius:16px; background:#3b167f; color:#fff; font:16px/1.5 system-ui,sans-serif;}
.sf-pair h3 {font-size:20px; margin:0 0 16px; color:#fff;}
.sf-pair .sf-row {display:flex; flex-wrap:wrap; align-items:center; gap:16px;}
.sf-pair input {box-sizing:border-box; width:10ch; min-width:0; max-width:100%; padding:8px 0;
  border:0; border-radius:4px; background:transparent; color:#ffe474;
  font:800 clamp(28px,5vw,42px)/1.2 ui-monospace,monospace; letter-spacing:.06em;}
.sf-pair button {min-height:48px; padding:12px 20px; border:0; border-radius:10px;
  background:#7af0de; color:#163c35; font:700 16px/1.5 system-ui,sans-serif; cursor:pointer;}
.sf-pair button:hover {background:#b0fff2;}
.sf-pair button:disabled {cursor:wait; opacity:.7;}
.sf-pair :focus-visible {outline:3px solid #fff; outline-offset:4px;}
.sf-pair input::selection {background:#ffe474; color:#3b167f;}
.sf-pair p {margin:16px 0 0; max-width:65ch; color:#ede2ff;}
.sf-pair [role=status] {margin-top:8px; min-height:24px; color:#fff;}
</style>
<section class="sf-pair" aria-label="Liên kết với StoryForge">
  <h3>Mã liên kết dùng một lần</h3>
  <div class="sf-row">
    <input aria-label="Mã liên kết" value="__SF_PAIR_CODE__" readonly spellcheck="false">
    <button type="button" onclick="(async (button) => {
      const root = button.closest('.sf-pair');
      const field = root.querySelector('input');
      const status = root.querySelector('[role=status]');
      button.disabled = true;
      let copied = false;
      try {
        field.focus(); field.select();
        copied = document.execCommand('copy');
      } catch (_) {}
      if (!copied) {
        try { await navigator.clipboard.writeText(field.value); copied = true; } catch (_) {}
      }
      button.disabled = false;
      if (copied) {
        button.textContent = 'Đã sao chép';
        status.textContent = 'Đã sao chép mã. Quay lại StoryForge để dán.';
        button.focus();
      } else {
        field.focus(); field.select();
        status.textContent = 'Trình duyệt chặn sao chép. Mã đã được chọn: nhấn Ctrl+C (Mac: ⌘C).';
      }
    })(this)">Sao chép mã</button>
  </div>
  <p>Dán mã vào <strong>Cài đặt → Máy AI cá nhân</strong> trong StoryForge,
  rồi xác nhận đúng tài khoản và máy.</p>
  <div role="status" aria-live="polite">Giữ riêng mã này. Đang chờ xác nhận trong StoryForge.</div>
</section>
""".replace('__SF_PAIR_CODE__', escape(str(code), quote=True))


def _sf_show_pairing_code(code):
    try:
        from IPython.display import HTML, display
        display(HTML(_sf_pairing_card(code)))
    except Exception:
        pass


_sf_last_wait_status = [None]

def _sf_wait_message(status):
    if status == _sf_last_wait_status[0]:
        return
    _sf_last_wait_status[0] = status
    labels = {'pending': 'Đang chờ Anh/Chị xác nhận trong StoryForge…',
              'approved': 'Đã xác nhận. StoryForge đang chuẩn bị kết nối…',
              'provisioning': 'Đang cấp đường kết nối riêng…',
              'slow_down': 'Máy chủ yêu cầu chờ lâu hơn trước lần kiểm tra tiếp theo…'}
    if status in labels:
        print(labels[status], flush=True)

try:
    if thao_tac == 'Xem trạng thái':
        start_colab.show_linked_panel(SF_LINKED_STATE)
    elif thao_tac == 'Chẩn đoán':
        start_colab.diagnostics(SF_LINKED_STATE)
    elif thao_tac == 'Dừng máy chủ':
        start_colab.stop(SF_LINKED_STATE)
        print('Đã dừng các tiến trình do notebook quản lý. Kết nối sẽ hết lease nếu không còn heartbeat.')
    else:
        if thao_tac == 'Khôi phục cấu hình trước':
            _sf_previous = SF_LINKED_STATE.get('previous_model_config')
            if not _sf_previous:
                raise ValueError('Chưa có cấu hình trước để khôi phục.')
            _sf_settings = {'model_choice': 'GGUF khác', 'device': 'GPU' if _sf_previous['device'] == 'gpu' else 'CPU',
                'parallel': _sf_previous.get('parallel', 1),
                'enable_thinking': bool(_sf_previous.get('enable_thinking', False)),
                **{key: _sf_previous[key] for key in ('context', 'max_tokens', 'repo', 'filename', 'revision')}}
        _sf_link = linked_connection.LinkedConnection(SF_ORIGIN, SF_LINKED_STATE)
        if thao_tac == 'Ghép nối / khởi động' and SF_LINKED_STATE.get('linked_credentials'):
            _sf_lease = SF_LINKED_STATE.get('linked_lease')
            _sf_stale = False
            if _sf_lease is not None:
                if not _sf_lease.is_active():
                    _sf_stale = True
                elif callable(getattr(_sf_lease, 'verify_live_lease', None)) and not _sf_lease.verify_live_lease():
                    _sf_stale = True
            if _sf_stale:
                print('Phiên kết nối trước đã bị thu hồi hoặc hết hạn trên StoryForge; đang dọn và tạo mã liên kết mới…', flush=True)
                _sf_lose = getattr(start_colab, '_lose_linked', None)
                if callable(_sf_lose):
                    _sf_lose(SF_LINKED_STATE, reason=getattr(_sf_lease, 'lost_reason', 'revoked') or 'revoked')
                else:
                    for _sf_k in ('linked_lease', 'linked_connection_id', 'linked_credentials',
                                  'linked_pair', 'key', 'gateway_hash', 'tunnel_provider', 'linked_started'):
                        SF_LINKED_STATE.pop(_sf_k, None)
        if thao_tac == 'Tạo mã liên kết mới':
            start_colab.prepare_new_link(SF_LINKED_STATE)
            _sf_pair = _sf_link.new_pair(label='Colab AI')
        else:
            _sf_pair = _sf_link.ensure_pair(label='Colab AI')
        if _sf_pair:
            _sf_show_pairing_code(_sf_pair['code'])
            print('Mã liên kết dùng một lần:', _sf_pair['code'], flush=True)
            print('Nhập mã này trong Cài đặt → Máy AI cá nhân của StoryForge và xác nhận rõ ràng.', flush=True)
        if SF_LINKED_STATE.get('linked_credentials'):
            _sf_credentials = SF_LINKED_STATE['linked_credentials']
        else:
            _sf_last_wait_status[0] = None
            print('Đang chờ xác nhận; lần chạy lại sẽ tiếp tục cùng mã liên kết.', flush=True)
            _sf_wait_progress = lambda _elapsed, _code: print(
                f'      Vẫn đang chờ nhập mã ({_code}) trên StoryForge · đã chờ {_elapsed}/600 giây…',
                flush=True)
            _sf_credentials = _sf_link.poll_until_claim(
                max_wait=600, on_status=_sf_wait_message, on_wait=_sf_wait_progress)
        print('Đã nhận quyền phiên trong bộ nhớ. Credential không được in ra notebook.', flush=True)
        start_colab.start_linked(SF_LINKED_STATE, _sf_credentials, SF_ORIGIN, settings=_sf_settings)
        start_colab.show_linked_panel(SF_LINKED_STATE)
        print('Ghi chú: Ô chạy sẽ tự dừng vòng xoay "Đang thực thi" để không khóa thao tác Colab; máy AI vẫn chạy nền.', flush=True)
        print('Muốn đổi model_AI, kiểm tra trạng thái hoặc chẩn đoán lỗi: chọn thao_tac bên trên rồi bấm Chạy lại ô.', flush=True)
except KeyboardInterrupt:
    print('Đã dừng lượt khởi động/tải model. Chạy lại ô để thử tiếp; tải lại trang không khởi động lại máy Colab.', flush=True)
    start_colab.show_linked_panel(SF_LINKED_STATE)
except Exception as _sf_error:
    _sf_runtime_err = getattr(start_colab, 'RuntimeActionError', None)
    _sf_safe_types = (ValueError, TimeoutError, RuntimeError, FileNotFoundError) + (
        (_sf_runtime_err,) if isinstance(_sf_runtime_err, type) else ()
    )
    if isinstance(_sf_error, _sf_safe_types) and type(_sf_error).__name__ != 'JSONDecodeError':
        import re as _sf_re
        _sf_safe_msg = _sf_re.sub(r'https?://\S+', '[URL]', str(_sf_error))
        _sf_safe_msg = _sf_re.sub(r'(?i)(bearer\s+)\S+', r'\1[ẩn]', _sf_safe_msg)
        print('Chưa áp dụng yêu cầu:', _sf_safe_msg)
    else:
        print('CHƯA SẴN SÀNG:', type(_sf_error).__name__)
        print('Chi tiết thô và nhật ký riêng tư được ẩn. Chọn Chẩn đoán để xem trạng thái tiến trình.')
    start_colab.show_linked_panel(SF_LINKED_STATE)
